In [ ]:
import duckdb
con = duckdb.connect('nyc311.db')

In [ ]:
con.execute("SELECT COUNT(*) FROM requests").fetchdf()

In [ ]:
con.execute("""
    SELECT complaint_type, 
           AVG(DATEDIFF('hour', CAST(created_date AS TIMESTAMP), CAST(closed_date AS TIMESTAMP))) as avg_hrs,
           COUNT(*) as n
    FROM requests
    WHERE closed_date IS NOT NULL
    GROUP BY complaint_type
    ORDER BY n DESC
    LIMIT 15
""").fetchdf()

In [ ]:
con.execute("""
    SELECT complaint_type,
           MIN(DATEDIFF('hour', CAST(created_date AS TIMESTAMP), CAST(closed_date AS TIMESTAMP))) as min_hrs,
           MAX(DATEDIFF('hour', CAST(created_date AS TIMESTAMP), CAST(closed_date AS TIMESTAMP))) as max_hrs,
           MEDIAN(DATEDIFF('hour', CAST(created_date AS TIMESTAMP), CAST(closed_date AS TIMESTAMP))) as median_hrs
    FROM requests
    WHERE closed_date IS NOT NULL
    GROUP BY complaint_type
    ORDER BY max_hrs DESC
    LIMIT 15
""").fetchdf()

In [ ]:
con.execute("""
    SELECT complaint_type,
           MEDIAN(DATEDIFF('hour', CAST(created_date AS TIMESTAMP), CAST(closed_date AS TIMESTAMP))) as median_hrs,
           AVG(DATEDIFF('hour', CAST(created_date AS TIMESTAMP), CAST(closed_date AS TIMESTAMP))) as mean_hrs,
           SUM(CASE WHEN DATEDIFF('hour', CAST(created_date AS TIMESTAMP), CAST(closed_date AS TIMESTAMP)) > 2160 THEN 1 ELSE 0 END) as over_90_days,
           COUNT(*) as total
    FROM requests
    WHERE closed_date IS NOT NULL
    GROUP BY complaint_type
    ORDER BY total DESC
""").fetchdf()

In [ ]:
con.execute("""
    SELECT unique_key, created_date, closed_date,
           DATEDIFF('hour', CAST(created_date AS TIMESTAMP), CAST(closed_date AS TIMESTAMP)) as hrs
    FROM requests
    WHERE complaint_type = 'UNSANITARY CONDITION'
    ORDER BY hrs DESC
    LIMIT 5
""").fetchdf()

In [ ]:
con.execute("""
    SELECT COUNT(*) as n_types_kept
    FROM (
        SELECT complaint_type, COUNT(*) as n
        FROM requests
        WHERE closed_date IS NOT NULL
        GROUP BY complaint_type
        HAVING COUNT(*) >= 30
    )
""").fetchdf()